# GPT-2 Inference Engine - Google Colab Setup

This notebook sets up the GPT-2 inference engine on Google Colab with T4 GPU acceleration.

**Runtime:** GPU (T4) | **Estimated time:** 15-20 minutes

## Step 1: Environment Setup

Check GPU availability and install build dependencies.

In [1]:
# Check GPU
!nvidia-smi

Fri Mar 27 07:38:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install build dependencies
!apt-get update && apt-get install -y build-essential cmake git wget curl htop

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restrict

In [3]:
# Check CUDA version
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## Step 2: Clone and Build GGML with CUDA Support

GGML provides the tensor computation backend with CUDA acceleration.

In [4]:
# Clone GGML (using master branch for compatibility)
%cd /content
!rm -rf ggml && git clone https://github.com/ggerganov/ggml.git --depth 1 ggml

/content
Cloning into 'ggml'...
remote: Enumerating objects: 1908, done.
remote: Counting objects: 100% (1908/1908), done.
remote: Compressing objects: 100% (1459/1459), done.
remote: Total 1908 (delta 493), reused 1479 (delta 434), pack-reused 0 (from 0)
Receiving objects: 100% (1908/1908), 2.91 MiB | 11.72 MiB/s, done.
Resolving deltas: 100% (493/493), done.


In [5]:
# Build GGML
# Note: Using standard CPU build first to ensure compatibility, then we can enable CUDA
%cd /content/ggml
!mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release && make -j$(nproc)

/content/ggml
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (f

In [6]:
# Verify GGML build
!cd /content/ggml/build && ls -la lib/

ls: cannot access 'lib/': No such file or directory


## Step 3: Clone Inference Engine

Copy your inference engine source files to the Colab environment.

In [7]:
# Option A: If using Google Drive (mount first)
# from google.colab import drive
# drive.mount('/content/gdrive')

# Option B: If uploading files directly - use the Files panel (left sidebar)
# Or clone from GitHub if hosted:
# !git clone https://github.com/YOUR_USERNAME/inference-engine.git

# For now, we'll create the project structure (only if not already cloned)
# Note: git clone in next cell will handle directory creation

In [8]:
# Clone the inference engine repo (includes CMakeLists.txt)
%cd /content
!rm -rf inference-engine && git clone https://github.com/nisbenz/inference-engine.git inference-engine

# If your repo is private or not yet pushed, manually create structure:
# !mkdir -p /content/inference-engine/src

# Verify CMakeLists.txt exists
!ls -la /content/inference-engine/

/content
Cloning into 'inference-engine'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 63 (delta 26), reused 50 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 47.79 KiB | 15.93 MiB/s, done.
Resolving deltas: 100% (26/26), done.
total 40
drwxr-xr-x 4 root root  4096 Mar 27 07:40 .
drwxr-xr-x 1 root root  4096 Mar 27 07:40 ..
-rw-r--r-- 1 root root   880 Mar 27 07:40 CMakeLists.txt
-rw-r--r-- 1 root root 11928 Mar 27 07:40 colab_gpt2_inference.ipynb
drwxr-xr-x 8 root root  4096 Mar 27 07:40 .git
-rw-r--r-- 1 root root  1064 Mar 27 07:40 LICENSE
-rw-r--r-- 1 root root    15 Mar 27 07:40 readme.md
drwxr-xr-x 2 root root  4096 Mar 27 07:40 src


In [9]:
# Upload your source files
# Use the Files panel to upload src/model.cpp, src/model.hpp, src/layers.cpp,
# src/layers.hpp, src/tokenizer.cpp, src/tokenizer.hpp, src/kv_cache.cpp,
# src/kv_cache.hpp, src/main.cpp

# Or run this cell after uploading via the Files panel
import os

src_dir = '/content/inference-engine/src'
print(f'Source files in {src_dir}:')
for f in os.listdir(src_dir):
    print(f'  {f}')

Source files in /content/inference-engine/src:
  layers.cpp
  model.cpp
  kv_cache.cpp
  tokenizer.cpp
  tokenizer.hpp
  layers.hpp
  main.cpp
  model.hpp
  kv_cache.hpp


## Step 4: Download GPT-2 Model (GGUF Format)

Download pre-converted GPT-2 GGUF model directly.

In [10]:
# Install HuggingFace Hub for downloading models
!pip install huggingface_hub -q

In [11]:
# Download GPT-2 GGUF directly from HuggingFace
# Using QuantFactory's GPT-2 GGUF model

%cd /content
!pip install huggingface_hub -q

from huggingface_hub import hf_hub_download
from huggingface_hub import list_repo_files

# List available GGUF files
print("Available files in QuantFactory/gpt2-GGUF:")
files = list_repo_files("QuantFactory/gpt2-GGUF")
for f in files:
    if f.endswith('.gguf'):
        print(f"  {f}")

/content
Available files in QuantFactory/gpt2-GGUF:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


  gpt2.Q2_K.gguf
  gpt2.Q3_K_L.gguf
  gpt2.Q3_K_M.gguf
  gpt2.Q3_K_S.gguf
  gpt2.Q4_0.gguf
  gpt2.Q4_1.gguf
  gpt2.Q4_K_M.gguf
  gpt2.Q4_K_S.gguf
  gpt2.Q5_0.gguf
  gpt2.Q5_1.gguf
  gpt2.Q5_K_M.gguf
  gpt2.Q5_K_S.gguf
  gpt2.Q6_K.gguf
  gpt2.Q8_0.gguf


In [12]:
# Download GPT-2 GGUF model (adjust filename based on available files above)
# Using Q4_K_M quantization for balance of speed and quality

print("Downloading GPT-2 GGUF model...")
model_path = hf_hub_download(
    repo_id="QuantFactory/gpt2-GGUF",
    filename="gpt2.Q4_K_M.gguf",
    repo_type="model",
    local_dir="/content/gpt2-model"
)
print(f"Model downloaded to: {model_path}")

gpt2.Q4_K_M.gguf:   0%|          | 0.00/113M [00:00<?, ?B/s]

Model downloaded to: /content/gpt2-model/gpt2.Q4_K_M.gguf


In [13]:
# Verify GGUF file
!ls -lh /content/gpt2-model/

total 108M
-rw-r--r-- 1 root root 108M Mar 27 07:41 gpt2.Q4_K_M.gguf


In [14]:
# Verify GGUF file
!ls -lh /content/gpt2-model/

total 108M
-rw-r--r-- 1 root root 108M Mar 27 07:41 gpt2.Q4_K_M.gguf


## Step 5: Update Source Code for GGUF Loading

The current `load_weights()` needs to be updated to parse GGUF format.

In [15]:
# Create updated model.cpp with GGUF loading support
# This replaces load_weights() with actual GGUF implementation

model_cpp_content = '''
// ... (rest of your existing model.cpp)

bool GPT2Model::load_weights(const std::string& path) {
    // Try GGUF format first
    struct gguf_context* ctx = gguf_load(path.c_str());
    if (ctx) {
        printf("Loaded GGUF model from: %s\\n", path.c_str());

        // Get tensor count
        int n_tensors = gguf_get_n_tensors(ctx);
        printf("Number of tensors: %d\\n", n_tensors);

        // Load each tensor
        for (int i = 0; i < n_tensors; i++) {
            const char* name = gguf_get_tensor_name(ctx, i);
            struct ggml_tensor* tensor = ggml_get_tensor(ctx_, name);
            if (tensor) {
                // Read tensor data from file
                // This is handled by ggml_init_from_file
            }
        }

        gguf_free(ctx);
        return true;
    }

    // Fallback to binary format
    return load_ggml_weights(path);
}
'''

print('GGUF loading code template created')
print('Note: You need to integrate this with your actual model.cpp')

GGUF loading code template created
Note: You need to integrate this with your actual model.cpp


## Step 6: Build Custom Inference Engine

Build your own inference engine with the updated source code.

In [16]:
# Build the inference engine
# Note: GGML_DIR must point to the ggml source directory so cmake can find ggml/ggml.h
!cd /content/inference-engine && mkdir -p build && cd build && cmake .. -DCMAKE_CUDA_PATH=/usr/local/cuda -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_DIR=/content/ggml && make -j$(nproc)

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Configuring done (4.7s)
-- Generating done (0.0s)
CMake Warning:
  Manually-specified variables were not used by the project:

    CMAKE_CUDA_PATH


-- Build files have been written

In [17]:
# Verify build
!cd /content/inference-engine/build && ls -la

total 44
drwxr-xr-x 3 root root  4096 Mar 27 07:41 .
drwxr-xr-x 5 root root  4096 Mar 27 07:41 ..
-rw-r--r-- 1 root root 15756 Mar 27 07:41 CMakeCache.txt
drwxr-xr-x 7 root root  4096 Mar 27 07:41 CMakeFiles
-rw-r--r-- 1 root root  2089 Mar 27 07:41 cmake_install.cmake
-rw-r--r-- 1 root root  8506 Mar 27 07:41 Makefile


## Step 7: Run Inference

Execute your inference engine.

In [18]:
# Run with your inference engine
!cd /content/inference-engine/build && ./gpt2 "Hello, world!" 50 0.8 40

/bin/bash: line 1: ./gpt2: No such file or directory


## Troubleshooting

### Common Issues:

1. **CUDA out of memory**: Reduce batch size or use smaller quantization (e.g., Q3_K_M instead of Q4_K_M)
2. **GGUF download fails**: Check HuggingFace connection, try alternative model repo
3. **Build errors**: Ensure all source files are uploaded to `/content/inference-engine/src/`
4. **Tensor mismatch**: GPT-2 Large (774M) config: 36 layers, 20 heads, 1280 hidden size

In [19]:
# Check GGUF model metadata
!pip install numpy sentencepiece -q
import struct

def read_gguf_header(path):
    with open(path, 'rb') as f:
        magic = struct.unpack('I', f.read(4))[0]
        print(f'Magic: {hex(magic)}')
        version = struct.unpack('I', f.read(4))[0]
        print(f'Version: {version}')

import glob
gguf_files = glob.glob('/content/gpt2-model/*.gguf')
if gguf_files:
    read_gguf_header(gguf_files[0])
else:
    print("No GGUF file found")

Magic: 0x46554747
Version: 3


## File Summary

After uploading your source files, the structure should be:
```
/content/
├── ggml/                    # GGML library (CUDA enabled)
├── inference-engine/        # Your project
│   ├── src/
│   │   ├── model.cpp
│   │   ├── model.hpp
│   │   ├── layers.cpp
│   │   ├── layers.hpp
│   │   ├── tokenizer.cpp
│   │   ├── tokenizer.hpp
│   │   ├── kv_cache.cpp
│   │   └── kv_cache.hpp
│   ├── build/
│   └── CMakeLists.txt
└── gpt2-model/              # GPT-2 GGUF model file (~500MB for Q4_K_M)
```

## Next Steps

1. Upload your source files to `/content/inference-engine/src/`
2. Update `model.cpp` to implement GGUF loading
3. Run cells from Step 6 onwards
4. For custom inference logic, modify the model classes